# LFP and Motion Energy


In [1]:
import sys
import importlib
from pathlib import Path
from dataclasses import replace

import matplotlib

try:
    ip = get_ipython()
except NameError:
    ip = None

if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "qt")
    except Exception as exc:
        print(f"Could not enable %matplotlib qt, using {matplotlib.get_backend()}: {exc}")
else:
    try:
        matplotlib.use("QtAgg", force=True)
    except Exception as exc:
        print(f"Could not enable QtAgg backend, using {matplotlib.get_backend()}: {exc}")

import matplotlib.pyplot as plt
import numpy as np

import lfp_motion_energy_utils as lme_utils
lme_utils = importlib.reload(lme_utils)
from lfp_motion_energy_utils import (
    amplitude_envelope_1d,
    CameraSyncSettings,
    benchmark_lfp_loading,
    build_lfp_motion_energy_cache,
    butter_filter_1d,
    channel_range,
    clean_motion_energy_regular_artifacts,
    compute_lfp_magnitude_and_me,
    compute_lfp_magnitude_and_me_from_cache,
    correlate_unit_rates_to_trace,
    decimate_spike_rate_result,
    decimate_for_lag,
    default_lfp_motion_energy_cache_path,
    default_lfp_motion_energy_trace_cache_path,
    find_lfp_motion_energy_trace_cache,
    future_trace_change,
    gaussian_smooth_rows,
    lagged_pearson,
    load_channel_average_lfp,
    load_kilosort_spikes,
    load_lfp_motion_energy_cache,
    load_lfp_motion_energy_trace_cache,
    load_motion_energy,
    load_recording,
    map_spike_rates_to_timebase,
    pearsonr_valid,
    percentile_minmax,
    print_recording_summary,
    subtract_static_channel_mean,
    unit_trace_lag_correlations,
    zscore,
)


In [2]:
# Dataset
EXPERIMENT_PATH = Path("/home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1")


In [3]:
# Helper functions
def corr_filter_settings() -> dict:
    cfg = {
        "channel_start": globals().get("CORR_CHANNEL_START", 0),
        "channel_stop": globals().get("CORR_CHANNEL_STOP", None),
        "channel_step": globals().get("CORR_CHANNEL_STEP", 1),
        "start_s": globals().get("CORR_START_S", 0.0),
        "duration_s": globals().get("CORR_DURATION_S", None),
        "chunk_seconds": globals().get("CORR_CHUNK_SECONDS", 60.0),
        "smooth_seconds": globals().get("CORR_SMOOTH_SECONDS", 0.0),
        "target_rate_hz": globals().get("TARGET_RATE_HZ", 250.0),
        "read_chunk_seconds": globals().get("LFP_READ_CHUNK_SECONDS", 1.0),
        "read_mode": globals().get("LFP_READ_MODE", "auto"),
        "bandpass_hz": globals().get("CORR_LFP_BANDPASS_HZ", None),
        "motion_energy_envelope": globals().get("CORR_ME_ENVELOPE", True),
        "me_artifact_filter": globals().get("CORR_ME_ARTIFACT_FILTER", globals().get("ME_ARTIFACT_FILTER", False)),
        "me_artifact_method": globals().get("CORR_ME_ARTIFACT_METHOD", globals().get("ME_ARTIFACT_METHOD", "fixed")),
        "me_artifact_diff_height": globals().get("CORR_ME_ARTIFACT_DIFF_HEIGHT", globals().get("ME_ARTIFACT_DIFF_HEIGHT", None)),
        "me_artifact_period_frames": globals().get("CORR_ME_ARTIFACT_PERIOD_FRAMES", globals().get("ME_ARTIFACT_PERIOD_FRAMES", 250)),
        "me_artifact_before_frames": globals().get("CORR_ME_ARTIFACT_BEFORE_FRAMES", globals().get("ME_ARTIFACT_BEFORE_FRAMES", 3)),
        "me_artifact_after_frames": globals().get("CORR_ME_ARTIFACT_AFTER_FRAMES", globals().get("ME_ARTIFACT_AFTER_FRAMES", 10)),
        "trace_cache": globals().get("CORR_TRACE_CACHE", True),
        "trace_cache_path": globals().get("CORR_TRACE_CACHE_PATH", None),
        "trace_cache_overwrite": globals().get("CORR_TRACE_CACHE_OVERWRITE", False),
        "trace_cache_allow_channel_mismatch": globals().get("CORR_TRACE_CACHE_ALLOW_CHANNEL_MISMATCH", False),
    }
    corr_stop = recording.lfp.n_lfp_channels if cfg["channel_stop"] is None else cfg["channel_stop"]
    return {
        "channels": tuple(int(ch) for ch in channel_range(cfg["channel_start"], corr_stop, cfg["channel_step"])),
        "start_s": float(cfg["start_s"]),
        "duration_s": None if cfg["duration_s"] is None else float(cfg["duration_s"]),
        "chunk_seconds": float(cfg["chunk_seconds"]),
        "smooth_seconds": float(cfg["smooth_seconds"]),
        "target_rate_hz": float(cfg["target_rate_hz"]),
        "read_chunk_seconds": float(cfg["read_chunk_seconds"]),
        "read_mode": str(cfg["read_mode"]),
        "bandpass_hz": None if cfg["bandpass_hz"] is None else tuple(float(v) for v in cfg["bandpass_hz"]),
        "motion_energy_envelope": bool(cfg["motion_energy_envelope"]),
        "me_artifact_filter": bool(cfg["me_artifact_filter"]),
        "me_artifact_method": str(cfg["me_artifact_method"]),
        "me_artifact_diff_height": None if cfg["me_artifact_diff_height"] is None else float(cfg["me_artifact_diff_height"]),
        "me_artifact_period_frames": int(cfg["me_artifact_period_frames"]),
        "me_artifact_before_frames": int(cfg["me_artifact_before_frames"]),
        "me_artifact_after_frames": int(cfg["me_artifact_after_frames"]),
        "trace_cache": bool(cfg["trace_cache"]),
        "trace_cache_path": None if cfg["trace_cache_path"] is None else str(cfg["trace_cache_path"]),
        "trace_cache_overwrite": bool(cfg["trace_cache_overwrite"]),
        "trace_cache_allow_channel_mismatch": bool(cfg["trace_cache_allow_channel_mismatch"]),
    }


def corr_recording_for_settings(settings: dict):
    from dataclasses import replace as dataclass_replace

    if not settings["me_artifact_filter"]:
        return recording, np.empty(0, dtype=np.int64)

    _me_path, me_raw, _me_attrs = lme_utils.load_motion_energy(recording.experiment_path)
    me_n = min(len(me_raw), len(recording.frame_video_seconds))
    if me_n != len(me_raw) or me_n != len(recording.frame_video_seconds):
        print(
            f"Corr ME/frame length mismatch; trimming ME/frame arrays to {me_n:,} "
            f"samples ({len(me_raw):,} ME, {len(recording.frame_video_seconds):,} frames)"
        )
    me_clean, artifact_frames = lme_utils.clean_motion_energy_regular_artifacts(
        me_raw[:me_n],
        method=settings["me_artifact_method"],
        diff_height=settings["me_artifact_diff_height"],
        period_frames=settings["me_artifact_period_frames"],
        before_frames=settings["me_artifact_before_frames"],
        after_frames=settings["me_artifact_after_frames"],
    )
    corr_recording = dataclass_replace(
        recording,
        motion_energy=me_clean.astype(np.float32, copy=False),
        me_attrs={
            **dict(recording.me_attrs),
            "artifact_filter_enabled": True,
            "artifact_filter_method": settings["me_artifact_method"],
            "artifact_filter_diff_height": settings["me_artifact_diff_height"],
            "artifact_filter_period_frames": settings["me_artifact_period_frames"],
            "artifact_filter_before_frames": settings["me_artifact_before_frames"],
            "artifact_filter_after_frames": settings["me_artifact_after_frames"],
            "artifact_filter_n_frames": int(len(artifact_frames)),
        },
        frame_abs_seconds=recording.frame_abs_seconds[:me_n],
        frame_video_seconds=recording.frame_video_seconds[:me_n],
    )
    return corr_recording, artifact_frames


def corr_trace_cache_range(settings: dict, corr_recording) -> tuple[float, float] | None:
    overlap_start_s, overlap_stop_s = corr_recording.lfp_video_overlap()
    start_s = max(float(settings["start_s"]), overlap_start_s)
    stop_s = overlap_stop_s if settings["duration_s"] is None else min(
        start_s + float(settings["duration_s"]),
        overlap_stop_s,
    )
    if stop_s <= start_s:
        return None
    return start_s, stop_s


def corr_trace_cache_path_for_settings(settings: dict, corr_recording) -> Path | None:
    if not settings["trace_cache"]:
        return None
    if settings["trace_cache_path"] is not None:
        return Path(settings["trace_cache_path"])
    cache_range = corr_trace_cache_range(settings, corr_recording)
    if cache_range is None:
        return None
    start_s, stop_s = cache_range
    return lme_utils.default_lfp_motion_energy_trace_cache_path(
        corr_recording,
        np.asarray(settings["channels"], dtype=np.int64),
        target_rate_hz=settings["target_rate_hz"],
        start_s=start_s,
        stop_s=stop_s,
        smooth_seconds=settings["smooth_seconds"],
        lfp_bandpass_hz=settings["bandpass_hz"],
        motion_energy_envelope=settings["motion_energy_envelope"],
    )


def find_corr_trace_cache_for_settings(settings: dict, corr_recording, trace_cache_path: Path | None) -> Path | None:
    if not settings["trace_cache"] or settings["trace_cache_overwrite"] or trace_cache_path is None:
        return None
    cache_range = corr_trace_cache_range(settings, corr_recording)
    if cache_range is None:
        return None
    start_s, stop_s = cache_range
    requested_channels = np.asarray(settings["channels"], dtype=np.int64)
    found_cache = lme_utils.find_lfp_motion_energy_trace_cache(
        corr_recording,
        requested_channels,
        target_rate_hz=settings["target_rate_hz"],
        start_s=start_s,
        stop_s=stop_s,
        smooth_seconds=settings["smooth_seconds"],
        lfp_bandpass_hz=settings["bandpass_hz"],
        motion_energy_envelope=settings["motion_energy_envelope"],
        cache_path=trace_cache_path,
    )
    if found_cache is not None:
        return found_cache
    if not settings["trace_cache_allow_channel_mismatch"]:
        return None
    return lme_utils.find_lfp_motion_energy_trace_cache(
        corr_recording,
        requested_channels,
        target_rate_hz=settings["target_rate_hz"],
        start_s=start_s,
        stop_s=stop_s,
        smooth_seconds=settings["smooth_seconds"],
        lfp_bandpass_hz=settings["bandpass_hz"],
        motion_energy_envelope=settings["motion_energy_envelope"],
        allow_channel_mismatch=True,
        search_other_channel_caches=True,
    )


def update_corr_trace_if_needed(force: bool = False):
    global corr_channels, corr_trace, corr_r, corr_valid, corr_lfp_label, corr_me_label, corr_trace_settings, corr_artifact_frames
    settings = corr_filter_settings()
    if force or "corr_trace" not in globals() or settings["trace_cache_overwrite"] or globals().get("corr_trace_settings") != settings:
        requested_corr_channels = np.asarray(settings["channels"], dtype=np.int64)
        corr_recording, corr_artifact_frames = corr_recording_for_settings(settings)
        if settings["me_artifact_filter"]:
            print(f"corr ME artifact removal interpolated {len(corr_artifact_frames):,} frame(s)")
        corr_trace_cache_path = corr_trace_cache_path_for_settings(settings, corr_recording)
        found_trace_cache = find_corr_trace_cache_for_settings(settings, corr_recording, corr_trace_cache_path)
        cache_range = corr_trace_cache_range(settings, corr_recording)
        if found_trace_cache is not None and cache_range is not None:
            cache_channels = lme_utils.load_lfp_motion_energy_trace_cache_channels(found_trace_cache)
            corr_channels = cache_channels
            start_s, stop_s = cache_range
            if np.array_equal(cache_channels, requested_corr_channels):
                print(f"using matching corr trace cache: {found_trace_cache}")
            else:
                print(
                    f"using corr trace cache with cached channels {lme_utils.channel_label(cache_channels)} "
                    f"instead of requested {lme_utils.channel_label(requested_corr_channels)}: {found_trace_cache}"
                )
            corr_trace = lme_utils.load_lfp_motion_energy_trace_cache(
                found_trace_cache,
                start_s=start_s,
                duration_s=stop_s - start_s,
            )
        else:
            corr_channels = requested_corr_channels
            if settings["trace_cache"]:
                if settings["trace_cache_overwrite"]:
                    print(f"corr trace cache target: {corr_trace_cache_path}; overwrite=True")
                else:
                    print(f"no suitable corr trace cache found; will rebuild from raw cache: {corr_trace_cache_path}")
            else:
                print("corr trace cache disabled")
            corr_trace = lme_utils.compute_lfp_magnitude_and_me(
                corr_recording,
                corr_channels,
                start_s=settings["start_s"],
                duration_s=settings["duration_s"],
                chunk_seconds=settings["chunk_seconds"],
                smooth_seconds=settings["smooth_seconds"],
                target_rate_hz=settings["target_rate_hz"],
                read_chunk_seconds=settings["read_chunk_seconds"],
                read_mode=settings["read_mode"],
                lfp_bandpass_hz=settings["bandpass_hz"],
                use_trace_cache=settings["trace_cache"],
                trace_cache_path=corr_trace_cache_path,
                trace_cache_overwrite=settings["trace_cache_overwrite"],
                motion_energy_envelope=settings["motion_energy_envelope"],
            )
        corr_r, corr_valid = pearsonr_valid(corr_trace.lfp_trace, corr_trace.motion_energy)
        lfp_filter_label = "" if settings["bandpass_hz"] is None else f" {settings['bandpass_hz'][0]:g}-{settings['bandpass_hz'][1]:g} Hz"
        me_kind = "motion-energy envelope" if settings["motion_energy_envelope"] else "raw motion energy"
        if settings["me_artifact_filter"]:
            me_kind = "artifact-cleaned " + me_kind
        corr_lfp_label = f"mean LFP envelope{lfp_filter_label}, z"
        corr_me_label = f"{me_kind}{lfp_filter_label}, z"
        corr_trace_settings = settings
        print(
            f"Pearson r = {corr_r:.4f} over {corr_valid.sum():,} valid samples "
            f"at {corr_trace.sample_rate_hz:.1f} Hz"
        )
        print(
            "Trace minima before z-scoring: "
            f"LFP={np.nanmin(corr_trace.lfp_trace):.4g}, "
            f"ME={np.nanmin(corr_trace.motion_energy):.4g}"
        )
    return corr_trace


In [4]:
# Load recording metadata, camera TTLs, and motion energy
CAMERA = CameraSyncSettings(
    channel=1,
    threshold_mv=3000.0,
    min_separation_samples=85,
    force_redetect=False,
)

ME_ARTIFACT_FILTER = False
ME_ARTIFACT_METHOD = "fixed"       # "fixed" or "peak_regular"
ME_ARTIFACT_DIFF_HEIGHT = None
ME_ARTIFACT_PERIOD_FRAMES = 250
ME_ARTIFACT_BEFORE_FRAMES = 3
ME_ARTIFACT_AFTER_FRAMES = 10

recording = load_recording(EXPERIMENT_PATH, camera=CAMERA)
if ME_ARTIFACT_FILTER:
    artifact_frames = recording.apply_motion_energy_artifact_filter(
        method=ME_ARTIFACT_METHOD,
        diff_height=ME_ARTIFACT_DIFF_HEIGHT,
        period_frames=ME_ARTIFACT_PERIOD_FRAMES,
        before_frames=ME_ARTIFACT_BEFORE_FRAMES,
        after_frames=ME_ARTIFACT_AFTER_FRAMES,
    )
    print(f"removed/interpolated {len(artifact_frames):,} regular motion-energy artifact frames")
print_recording_summary(recording)


nChan: 385, nFileSamp: 19877151
loaded 591,868 cached camera TTLs from /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/CAM_TTLs.npy
nidq: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/20260714_ant_g0/20260714_ant_g0_t0.nidq.bin
lfp:  /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/20260714_ant_g0/20260714_ant_g0_imec0/20260714_ant_g0_t0.imec0.lf.bin
me:   /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/cam1_2026-07-14-11-05-52.avi.me
nidq rate=10593.400 Hz; lfp rate=2500.000 Hz
frames=591,868; video duration=132.31 min
lfp channels=384; lfp duration=132.51 min


In [18]:
# Build/load reusable 250 Hz LFP/ME cache
LFP_CACHE_BUILD = True
LFP_CACHE_LOAD = True
LFP_CACHE_OVERWRITE = False
LFP_CACHE_TARGET_RATE_HZ = 250.0
LFP_CACHE_CHANNEL_START = 0
LFP_CACHE_CHANNEL_STOP = None
LFP_CACHE_CHANNEL_STEP = 1
LFP_CACHE_START_S = 0.0
LFP_CACHE_DURATION_S = None
LFP_CACHE_CHUNK_SECONDS = 60.0
LFP_CACHE_PATH = None
LFP_READ_CHUNK_SECONDS = 1.0
LFP_READ_MODE = "auto"

lfp_cache_stop = recording.lfp.n_lfp_channels if LFP_CACHE_CHANNEL_STOP is None else LFP_CACHE_CHANNEL_STOP
lfp_cache_channels = channel_range(
    LFP_CACHE_CHANNEL_START,
    lfp_cache_stop,
    LFP_CACHE_CHANNEL_STEP,
)
lfp_cache_path = (
    default_lfp_motion_energy_cache_path(
        recording,
        lfp_cache_channels,
        target_rate_hz=LFP_CACHE_TARGET_RATE_HZ,
    )
    if LFP_CACHE_PATH is None
    else Path(LFP_CACHE_PATH)
)

if LFP_CACHE_BUILD:
    lfp_cache_path = build_lfp_motion_energy_cache(
        recording,
        lfp_cache_path,
        lfp_cache_channels,
        target_rate_hz=LFP_CACHE_TARGET_RATE_HZ,
        start_s=LFP_CACHE_START_S,
        duration_s=LFP_CACHE_DURATION_S,
        chunk_seconds=LFP_CACHE_CHUNK_SECONDS,
        read_chunk_seconds=LFP_READ_CHUNK_SECONDS,
        read_mode=LFP_READ_MODE,
        overwrite=LFP_CACHE_OVERWRITE,
    )
elif lfp_cache_path.exists():
    print(f"LFP/ME cache exists: {lfp_cache_path}")
else:
    print(f"No LFP/ME cache yet: {lfp_cache_path}")
    print("Set LFP_CACHE_BUILD=True and rerun this cell to create it.")

lfp_me_cache = None
if LFP_CACHE_LOAD and lfp_cache_path.exists():
    lfp_me_cache = load_lfp_motion_energy_cache(
        lfp_cache_path,
        start_s=LFP_CACHE_START_S,
        duration_s=LFP_CACHE_DURATION_S,
        channels=lfp_cache_channels,
    )
    print(
        f"loaded raw cache window: {len(lfp_me_cache.channels)} channel(s), "
        f"{len(lfp_me_cache.t_video_s):,} samples at {lfp_me_cache.sample_rate_hz:.1f} Hz"
    )


using existing LFP/ME cache: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
loaded raw cache window: 384 channel(s), 1,984,710 samples at 250.0 Hz


In [19]:
# Tune motion-energy artifact filter
TUNE_ME_START_FRAME = 0
TUNE_ME_N_FRAMES = 50000
TUNE_ME_METHOD = ME_ARTIFACT_METHOD
TUNE_ME_DIFF_HEIGHT = ME_ARTIFACT_DIFF_HEIGHT
TUNE_ME_PERIOD_FRAMES = ME_ARTIFACT_PERIOD_FRAMES
TUNE_ME_BEFORE_FRAMES = ME_ARTIFACT_BEFORE_FRAMES
TUNE_ME_AFTER_FRAMES = ME_ARTIFACT_AFTER_FRAMES

_me_path, tune_me_raw, _me_attrs = load_motion_energy(EXPERIMENT_PATH)
tune_me_clean, tune_artifact_frames = clean_motion_energy_regular_artifacts(
    tune_me_raw,
    method=TUNE_ME_METHOD,
    diff_height=TUNE_ME_DIFF_HEIGHT,
    period_frames=TUNE_ME_PERIOD_FRAMES,
    before_frames=TUNE_ME_BEFORE_FRAMES,
    after_frames=TUNE_ME_AFTER_FRAMES,
)

tune_stop_frame = min(len(tune_me_raw), TUNE_ME_START_FRAME + TUNE_ME_N_FRAMES)
tune_window = slice(TUNE_ME_START_FRAME, tune_stop_frame)
tune_x = np.arange(TUNE_ME_START_FRAME, tune_stop_frame)
tune_in_window = tune_artifact_frames[
    (tune_artifact_frames >= TUNE_ME_START_FRAME) & (tune_artifact_frames < tune_stop_frame)
]

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
axes[0].plot(tune_x, tune_me_raw[tune_window], lw=0.7, label="raw ME")
axes[0].plot(tune_x, tune_me_clean[tune_window], lw=0.9, label="cleaned ME")
if tune_in_window.size:
    axes[0].scatter(tune_in_window, tune_me_raw[tune_in_window], s=18, color="tab:red", label="artifact frames")
axes[0].set_ylabel("motion energy")
axes[0].set_title(
    f"ME artifact tuning: method={TUNE_ME_METHOD}, artifacts={len(tune_artifact_frames):,}, "
    f"window={TUNE_ME_BEFORE_FRAMES} before/{TUNE_ME_AFTER_FRAMES} after"
)
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(tune_x, tune_me_raw[tune_window] - tune_me_clean[tune_window], lw=0.7)
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].set_ylabel("raw - cleaned")
axes[1].grid(alpha=0.25)

tune_diff = np.diff(tune_me_raw.astype(np.float64, copy=False))
tune_diff_x = np.arange(TUNE_ME_START_FRAME, max(TUNE_ME_START_FRAME, tune_stop_frame - 1))
axes[2].plot(tune_diff_x, tune_diff[tune_window.start : max(tune_window.start, tune_window.stop - 1)], lw=0.7)
if TUNE_ME_METHOD == "peak_regular":
    tune_height = TUNE_ME_DIFF_HEIGHT
    if tune_height is None:
        finite_diff = tune_diff[np.isfinite(tune_diff)]
        tune_height = float(np.nanmedian(finite_diff) + 8.0 * np.nanstd(finite_diff))
    axes[2].axhline(tune_height, color="tab:red", lw=0.9, label=f"diff height={tune_height:.3g}")
    axes[2].legend()
axes[2].set_xlabel("frame")
axes[2].set_ylabel("diff(ME)")
axes[2].grid(alpha=0.25)
plt.show()

if tune_artifact_frames.size > 1:
    print(
        f"artifact frames: n={len(tune_artifact_frames):,}, "
        f"median spacing={np.median(np.diff(tune_artifact_frames)):.1f} frames, "
        f"first={tune_artifact_frames[:10]}"
    )
else:
    print(f"artifact frames: n={len(tune_artifact_frames):,}, first={tune_artifact_frames[:10]}")


artifact frames: n=2,368, median spacing=250.0 frames, first=[   0  250  500  750 1000 1250 1500 1750 2000 2250]


In [6]:
# Load a chunk of LFP near target rate and plot as an image
LFP_IMAGE_START_S = 0.0
LFP_IMAGE_DURATION_S = 50.0
LFP_IMAGE_CHANNEL_START = 0
LFP_IMAGE_CHANNEL_STOP = 260
LFP_IMAGE_CHANNEL_STEP = 2
TARGET_RATE_HZ = 250.0
LFP_READ_CHUNK_SECONDS = 1.0
LFP_READ_MODE = "direct"
LFP_IMAGE_USE_CACHE = True
LFP_IMAGE_CLOSE_OLD_FIGURES = False

image_channels = channel_range(
    LFP_IMAGE_CHANNEL_START,
    LFP_IMAGE_CHANNEL_STOP,
    LFP_IMAGE_CHANNEL_STEP,
)
importlib.reload(lme_utils)

lfp_image_cache_path = None
if LFP_IMAGE_USE_CACHE:
    lfp_image_cache_path = lme_utils.find_lfp_motion_energy_cache(
        recording,
        image_channels,
        target_rate_hz=TARGET_RATE_HZ,
        start_s=LFP_IMAGE_START_S,
        stop_s=LFP_IMAGE_START_S + LFP_IMAGE_DURATION_S,
    )

if lfp_image_cache_path is not None:
    print(f"using cached LFP image data: {lfp_image_cache_path}")
    lfp_image = lme_utils.load_lfp_motion_energy_cache(
        lfp_image_cache_path,
        start_s=LFP_IMAGE_START_S,
        duration_s=LFP_IMAGE_DURATION_S,
        channels=image_channels,
    )
else:
    print("LFP image cache not found; loading directly from raw LF file")
    lfp_image = recording.load_lfp_downsampled(
        LFP_IMAGE_START_S,
        LFP_IMAGE_DURATION_S,
        image_channels,
        target_rate_hz=TARGET_RATE_HZ,
        read_chunk_seconds=LFP_READ_CHUNK_SECONDS,
        read_mode=LFP_READ_MODE,
    )
if lfp_image.t_video_s.size == 0:
    raise ValueError("No LFP samples in requested image window")

lfp_demeaned = subtract_static_channel_mean(lfp_image.lfp_uv)
print(
    f"loaded LFP image array: {lfp_demeaned.shape[0]} channel(s) x "
    f"{lfp_demeaned.shape[1]:,} sample(s), {lfp_demeaned.nbytes / 1024**2:.2f} MiB, "
    f"sample_rate={lfp_image.sample_rate_hz:.1f} Hz"
)

vlim = np.nanpercentile(np.abs(lfp_demeaned), 99)
if not np.isfinite(vlim) or vlim <= 0:
    vlim = 1.0
if LFP_IMAGE_CLOSE_OLD_FIGURES:
    plt.close("all")
plt.ion()
fig, ax = plt.subplots(figsize=(13, 7), constrained_layout=True)
im = ax.imshow(
    lfp_demeaned,
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=-vlim,
    vmax=vlim,
    extent=[lfp_image.t_video_s[0], lfp_image.t_video_s[-1], image_channels[-1], image_channels[0]],
)
ax.set_title(
    f"LFP at {lfp_image.sample_rate_hz:.1f} Hz, static channel-mean subtracted, "
    f"{LFP_IMAGE_START_S:.1f}-{LFP_IMAGE_START_S + LFP_IMAGE_DURATION_S:.1f} s"
)
ax.set_xlabel("video time (s)")
ax.set_ylabel("saved LF channel")
fig.colorbar(im, ax=ax, label="mean-subtracted LFP (uV)")
fig.canvas.draw_idle()
plt.show(block=False)


using cached LFP image data: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
loaded LFP image array: 130 channel(s) x 12,500 sample(s), 6.20 MiB, sample_rate=250.0 Hz


In [5]:
# Plot cached LFP magnitude and motion energy only, without Kilosort units
NO_UNIT_LFP_CHANNELS = list(range(50, 52))
NO_UNIT_PLOT_START_MIN = 0.0
NO_UNIT_PLOT_DURATION_MIN = None  # None plots all available LFP/ME samples after START_MIN
NO_UNIT_LFP_BANDPASS_HZ = None
NO_UNIT_ME_ENVELOPE = False  # False plots raw current ME; True plots ME Hilbert envelope
NO_UNIT_ME_ARTIFACT_FILTER = True
NO_UNIT_ME_ARTIFACT_METHOD = globals().get("ME_ARTIFACT_METHOD", "fixed")
NO_UNIT_ME_ARTIFACT_DIFF_HEIGHT = globals().get("ME_ARTIFACT_DIFF_HEIGHT", None)
NO_UNIT_ME_ARTIFACT_PERIOD_FRAMES = globals().get("ME_ARTIFACT_PERIOD_FRAMES", 250)
NO_UNIT_ME_ARTIFACT_BEFORE_FRAMES = globals().get("ME_ARTIFACT_BEFORE_FRAMES", 3)
NO_UNIT_ME_ARTIFACT_AFTER_FRAMES = globals().get("ME_ARTIFACT_AFTER_FRAMES", 10)
NO_UNIT_SMOOTH_SECONDS = 0.0
NO_UNIT_TARGET_RATE_HZ = 250.0
NO_UNIT_CACHE_CHUNK_SECONDS = 60.0
NO_UNIT_TRACE_CACHE = True
NO_UNIT_TRACE_CACHE_PATH = None
NO_UNIT_TRACE_CACHE_OVERWRITE = True

from dataclasses import replace
from time import perf_counter
import lfp_motion_energy_utils as lme_utils
lme_utils = importlib.reload(lme_utils)

no_unit_channels = np.asarray(NO_UNIT_LFP_CHANNELS, dtype=np.int64)
if no_unit_channels.size == 0:
    raise ValueError("NO_UNIT_LFP_CHANNELS must contain at least one LF channel")

no_unit_start_s = NO_UNIT_PLOT_START_MIN * 60.0
no_unit_duration_s = None if NO_UNIT_PLOT_DURATION_MIN is None else NO_UNIT_PLOT_DURATION_MIN * 60.0
no_unit_window_label = "full available window" if no_unit_duration_s is None else f"{NO_UNIT_PLOT_DURATION_MIN:g} min window"

no_unit_recording = recording
no_unit_artifact_frames = np.empty(0, dtype=np.int64)
if NO_UNIT_ME_ARTIFACT_FILTER:
    _no_unit_me_path, no_unit_me_raw, _no_unit_me_attrs = lme_utils.load_motion_energy(recording.experiment_path)
    no_unit_me_n = min(len(no_unit_me_raw), len(recording.frame_video_seconds))
    if no_unit_me_n != len(recording.frame_video_seconds) or no_unit_me_n != len(no_unit_me_raw):
        print(
            f"No-unit ME/frame length mismatch; trimming ME/frame arrays to {no_unit_me_n:,} "
            f"samples ({len(no_unit_me_raw):,} ME, {len(recording.frame_video_seconds):,} frames)"
        )
    no_unit_me_clean, no_unit_artifact_frames = lme_utils.clean_motion_energy_regular_artifacts(
        no_unit_me_raw[:no_unit_me_n],
        method=NO_UNIT_ME_ARTIFACT_METHOD,
        diff_height=NO_UNIT_ME_ARTIFACT_DIFF_HEIGHT,
        period_frames=NO_UNIT_ME_ARTIFACT_PERIOD_FRAMES,
        before_frames=NO_UNIT_ME_ARTIFACT_BEFORE_FRAMES,
        after_frames=NO_UNIT_ME_ARTIFACT_AFTER_FRAMES,
    )
    no_unit_recording = replace(
        recording,
        motion_energy=no_unit_me_clean.astype(np.float32, copy=False),
        me_attrs={
            **dict(recording.me_attrs),
            "artifact_filter_enabled": True,
            "artifact_filter_method": NO_UNIT_ME_ARTIFACT_METHOD,
            "artifact_filter_diff_height": NO_UNIT_ME_ARTIFACT_DIFF_HEIGHT,
            "artifact_filter_period_frames": int(NO_UNIT_ME_ARTIFACT_PERIOD_FRAMES),
            "artifact_filter_before_frames": int(NO_UNIT_ME_ARTIFACT_BEFORE_FRAMES),
            "artifact_filter_after_frames": int(NO_UNIT_ME_ARTIFACT_AFTER_FRAMES),
            "artifact_filter_n_frames": int(len(no_unit_artifact_frames)),
        },
        frame_abs_seconds=recording.frame_abs_seconds[:no_unit_me_n],
        frame_video_seconds=recording.frame_video_seconds[:no_unit_me_n],
    )

if NO_UNIT_TRACE_CACHE_PATH is None:
    no_unit_trace_cache_path = (
        recording.experiment_path
        / f"lfp_me_envelope_cache_{float(NO_UNIT_TARGET_RATE_HZ):g}hz_{lme_utils.channel_label(no_unit_channels)}.h5"
    )
else:
    no_unit_trace_cache_path = Path(NO_UNIT_TRACE_CACHE_PATH)

no_unit_trace_settings = {
    "channels": tuple(int(ch) for ch in no_unit_channels),
    "start_s": float(no_unit_start_s),
    "duration_s": None if no_unit_duration_s is None else float(no_unit_duration_s),
    "bandpass_hz": None if NO_UNIT_LFP_BANDPASS_HZ is None else tuple(float(v) for v in NO_UNIT_LFP_BANDPASS_HZ),
    "me_envelope": bool(NO_UNIT_ME_ENVELOPE),
    "me_artifact_filter": bool(NO_UNIT_ME_ARTIFACT_FILTER),
    "me_artifact_method": str(NO_UNIT_ME_ARTIFACT_METHOD),
    "me_artifact_diff_height": None if NO_UNIT_ME_ARTIFACT_DIFF_HEIGHT is None else float(NO_UNIT_ME_ARTIFACT_DIFF_HEIGHT),
    "me_artifact_period_frames": int(NO_UNIT_ME_ARTIFACT_PERIOD_FRAMES),
    "me_artifact_before_frames": int(NO_UNIT_ME_ARTIFACT_BEFORE_FRAMES),
    "me_artifact_after_frames": int(NO_UNIT_ME_ARTIFACT_AFTER_FRAMES),
    "smooth_seconds": float(NO_UNIT_SMOOTH_SECONDS),
    "target_rate_hz": float(NO_UNIT_TARGET_RATE_HZ),
    "chunk_seconds": float(NO_UNIT_CACHE_CHUNK_SECONDS),
    "trace_cache": bool(NO_UNIT_TRACE_CACHE),
    "trace_cache_path": str(no_unit_trace_cache_path) if NO_UNIT_TRACE_CACHE else None,
    "trace_cache_overwrite": bool(NO_UNIT_TRACE_CACHE_OVERWRITE),
}

no_unit_t0 = perf_counter()
print(
    f"no-unit trace cache: {no_unit_trace_cache_path if NO_UNIT_TRACE_CACHE else 'disabled'}; "
    f"overwrite={NO_UNIT_TRACE_CACHE_OVERWRITE}; bandpass={NO_UNIT_LFP_BANDPASS_HZ}; "
    f"ME envelope={NO_UNIT_ME_ENVELOPE}; ME artifact filter={NO_UNIT_ME_ARTIFACT_FILTER}; "
    f"smooth={NO_UNIT_SMOOTH_SECONDS:g}s"
)
if NO_UNIT_ME_ARTIFACT_FILTER:
    print(f"no-unit ME artifact removal interpolated {len(no_unit_artifact_frames):,} frame(s)")
if (not NO_UNIT_TRACE_CACHE_OVERWRITE) and globals().get("_no_unit_trace_settings") == no_unit_trace_settings and "no_unit_trace" in globals():
    print("reusing existing no_unit_trace for unchanged settings")
else:
    no_unit_trace = lme_utils.compute_lfp_magnitude_and_me(
        no_unit_recording,
        no_unit_channels,
        start_s=no_unit_start_s,
        duration_s=no_unit_duration_s,
        chunk_seconds=NO_UNIT_CACHE_CHUNK_SECONDS,
        smooth_seconds=NO_UNIT_SMOOTH_SECONDS,
        target_rate_hz=NO_UNIT_TARGET_RATE_HZ,
        read_chunk_seconds=1.0,
        read_mode="auto",
        lfp_bandpass_hz=NO_UNIT_LFP_BANDPASS_HZ,
        use_cache=True,
        use_trace_cache=NO_UNIT_TRACE_CACHE,
        trace_cache_path=no_unit_trace_cache_path if NO_UNIT_TRACE_CACHE else None,
        trace_cache_overwrite=NO_UNIT_TRACE_CACHE_OVERWRITE,
        motion_energy_envelope=NO_UNIT_ME_ENVELOPE,
    )
    _no_unit_trace_settings = no_unit_trace_settings
    print(
        f"loaded/computed no-unit envelope trace in {perf_counter() - no_unit_t0:.1f} s; "
        f"{no_unit_trace.t_video_s.size:,} samples at {no_unit_trace.sample_rate_hz:.1f} Hz"
    )
if no_unit_trace.t_video_s.size == 0:
    raise ValueError("No samples in requested LFP/ME-only plot window")

no_unit_t0 = perf_counter()
no_unit_lfp_norm = lme_utils.percentile_minmax(
    no_unit_trace.lfp_trace,
    lower_percentile=1.0,
    upper_percentile=99.0,
)
no_unit_me_norm = lme_utils.percentile_minmax(
    no_unit_trace.motion_energy,
    lower_percentile=1.0,
    upper_percentile=99.0,
)
no_unit_r, no_unit_valid = lme_utils.pearsonr_valid(no_unit_trace.lfp_trace, no_unit_trace.motion_energy)
print(f"normalized/plotted no-unit traces in {perf_counter() - no_unit_t0:.1f} s")

no_unit_max_plot_points = 100_000
no_unit_plot_step = max(1, int(np.ceil(no_unit_trace.t_video_s.size / no_unit_max_plot_points)))
no_unit_plot_idx = np.arange(no_unit_trace.t_video_s.size)[::no_unit_plot_step]
no_unit_channel_label = lme_utils.channel_label(no_unit_channels)
no_unit_me_label = "motion-energy envelope" if NO_UNIT_ME_ENVELOPE else "artifact-cleaned raw motion energy" if NO_UNIT_ME_ARTIFACT_FILTER else "raw motion energy"

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
ax.plot(
    no_unit_trace.t_video_s[no_unit_plot_idx] / 60.0,
    no_unit_lfp_norm[no_unit_plot_idx],
    label="LFP envelope, 1-99% min-max",
    lw=0.9,
)
ax.plot(
    no_unit_trace.t_video_s[no_unit_plot_idx] / 60.0,
    no_unit_me_norm[no_unit_plot_idx],
    label=f"{no_unit_me_label}, 1-99% min-max",
    lw=0.9,
    color="tab:orange",
)
ax.set_title(
    f"LFP envelope and motion energy only, {no_unit_window_label}, "
    f"LF {no_unit_channel_label}, {no_unit_me_label} (r={no_unit_r:+.3f}, n={no_unit_valid.sum():,})"
)
ax.set_xlim(no_unit_trace.t_video_s[0] / 60.0, no_unit_trace.t_video_s[-1] / 60.0)
ax.set_ylim(-1, 3.03)
ax.set_xlabel("video time (min)")
ax.set_ylabel("1-99% min-max normalized envelope")
ax.legend()
ax.grid(alpha=0.25)
plt.show()



no-unit trace cache: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_envelope_cache_250hz_ch050-051.h5; overwrite=True; bandpass=None; ME envelope=False; ME artifact filter=True; smooth=0s
no-unit ME artifact removal interpolated 2,368 frame(s)
LFP/ME envelope cache overwrite enabled; skipping processed-cache load
using cached raw LFP/ME data: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5; recomputing envelopes
resampled motion energy from current recording state
computing cached LFP envelope and raw motion energy without bandpass filtering
saved LFP/ME envelope cache: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_envelope_cache_250hz_ch050-051.h5 (17.95 MiB)
loaded/computed no-unit envelope trace in 5.2 s; 1,984,710 samples at 250.0 Hz
normalized/plotted no-unit traces in 0.1 s


In [6]:
# Load Kilosort spikes and map unit rates onto the LFP/ME timebase.
# Passing EXPERIMENT_PATH lets the loader prefer EXPERIMENT_PATH/kilosort4.
# It falls back to a unique nested Kilosort result folder for older runs.
KILOSORT_PATH = EXPERIMENT_PATH
KILOSORT_INCLUDE_LABELS = "all"     # use "all" to include all clusters
KILOSORT_EXCLUDE_LABELS = None
SPIKE_RATE_SMOOTH_SECONDS = 0.02

# Load the LFP/ME correlation trace from the processed cache when it matches.
# If that cache is missing or has different ME cleanup/filter settings, rebuild it from the raw LFP/ME cache.
CORR_TRACE_CACHE = True
CORR_TRACE_CACHE_PATH = None
CORR_TRACE_CACHE_OVERWRITE = False
CORR_TRACE_CACHE_ALLOW_CHANNEL_MISMATCH = True
CORR_ME_ENVELOPE = globals().get("NO_UNIT_ME_ENVELOPE", False)
CORR_ME_ARTIFACT_FILTER = globals().get("NO_UNIT_ME_ARTIFACT_FILTER", True)
CORR_ME_ARTIFACT_METHOD = globals().get("NO_UNIT_ME_ARTIFACT_METHOD", globals().get("ME_ARTIFACT_METHOD", "fixed"))
CORR_ME_ARTIFACT_DIFF_HEIGHT = globals().get("NO_UNIT_ME_ARTIFACT_DIFF_HEIGHT", globals().get("ME_ARTIFACT_DIFF_HEIGHT", None))
CORR_ME_ARTIFACT_PERIOD_FRAMES = globals().get("NO_UNIT_ME_ARTIFACT_PERIOD_FRAMES", globals().get("ME_ARTIFACT_PERIOD_FRAMES", 250))
CORR_ME_ARTIFACT_BEFORE_FRAMES = globals().get("NO_UNIT_ME_ARTIFACT_BEFORE_FRAMES", globals().get("ME_ARTIFACT_BEFORE_FRAMES", 3))
CORR_ME_ARTIFACT_AFTER_FRAMES = globals().get("NO_UNIT_ME_ARTIFACT_AFTER_FRAMES", globals().get("ME_ARTIFACT_AFTER_FRAMES", 10))

lme_utils = importlib.reload(lme_utils)
update_corr_trace_if_needed(force=True)
CORR_TRACE_CACHE_OVERWRITE = False
if "corr_trace_settings" in globals():
    corr_trace_settings = dict(corr_trace_settings)
    corr_trace_settings["trace_cache_overwrite"] = False
kilosort_spikes = load_kilosort_spikes(
    KILOSORT_PATH,
    recording=recording,
    include_labels=KILOSORT_INCLUDE_LABELS,
    exclude_labels=KILOSORT_EXCLUDE_LABELS,
)
spike_rate = map_spike_rates_to_timebase(
    kilosort_spikes,
    corr_trace.t_video_s,
    smoothing_sigma_s=SPIKE_RATE_SMOOTH_SECONDS,
)
if kilosort_spikes.sorted_binary_start_video_seconds is not None:
    spike_sorted_mask = corr_trace.t_video_s >= kilosort_spikes.sorted_binary_start_video_seconds
else:
    spike_sorted_mask = np.ones(corr_trace.t_video_s.shape, dtype=bool)
if kilosort_spikes.sorted_binary_stop_video_seconds is not None:
    spike_sorted_mask &= corr_trace.t_video_s < kilosort_spikes.sorted_binary_stop_video_seconds

spike_lfp_r = correlate_unit_rates_to_trace(
    spike_rate,
    corr_trace.lfp_trace,
    valid_mask=spike_sorted_mask,
)
spike_me_r = correlate_unit_rates_to_trace(
    spike_rate,
    corr_trace.motion_energy,
    valid_mask=spike_sorted_mask,
)
print(
    f"Loaded {len(kilosort_spikes.unit_ids):,} Kilosort units from {kilosort_spikes.path}; "
    f"{len(kilosort_spikes.spike_times_samples):,} spikes, "
    f"{kilosort_spikes.sample_rate_hz:g} Hz"
)

def fmt_optional_seconds(value):
    return "open" if value is None else f"{value:.3f}"

if (
    kilosort_spikes.sorted_binary_start_video_seconds is not None
    or kilosort_spikes.sorted_binary_stop_video_seconds is not None
):
    print(
        "Kilosort sorted window on video axis: "
        f"{fmt_optional_seconds(kilosort_spikes.sorted_binary_start_video_seconds)}-"
        f"{fmt_optional_seconds(kilosort_spikes.sorted_binary_stop_video_seconds)} s; "
        f"overlap samples on corr_trace={spike_sorted_mask.sum():,}"
    )
print(
    f"Mapped spike rates: {spike_rate.spike_rate_hz.shape[0]:,} unit(s) x "
    f"{spike_rate.spike_rate_hz.shape[1]:,} samples at {corr_trace.sample_rate_hz:.1f} Hz; "
    f"smoothing sigma={SPIKE_RATE_SMOOTH_SECONDS:g}s"
)

rank_metric = np.maximum(
    np.nan_to_num(np.abs(spike_lfp_r), nan=-np.inf),
    np.nan_to_num(np.abs(spike_me_r), nan=-np.inf),
)
ranked_units = np.argsort(rank_metric)[::-1]
print("Top unit-rate correlations:")
for rank_idx in ranked_units[:10]:
    unit_id = int(spike_rate.unit_ids[rank_idx])
    label = spike_rate.unit_labels[rank_idx] or "unlabeled"
    mean_rate = float(np.nanmean(spike_rate.spike_rate_hz[rank_idx, spike_sorted_mask]))
    print(
        f"  unit {unit_id:>4d} ({label:>4s}): "
        f"r_lfp={spike_lfp_r[rank_idx]:+.3f}, "
        f"r_me={spike_me_r[rank_idx]:+.3f}, "
        f"mean_rate={mean_rate:.2f} Hz"
    )


corr ME artifact removal interpolated 2,368 frame(s)
using corr trace cache with cached channels ch050-051 instead of requested ch000-383: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_envelope_cache_250hz_ch050-051.h5
Pearson r = 0.2348 over 1,984,710 valid samples at 250.0 Hz
Trace minima before z-scoring: LFP=1.282, ME=7097
Loaded 515 Kilosort units from /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/20260714_ant_g0/kilosort4; 29,599,550 spikes, 30000 Hz
Kilosort sorted window on video axis: -10.043-open s; overlap samples on corr_trace=1,984,710
Mapped spike rates: 515 unit(s) x 1,984,710 samples at 250.0 Hz; smoothing sigma=0.02s
Top unit-rate correlations:
  unit  450 ( mua): r_lfp=-0.352, r_me=-0.398, mean_rate=252.60 Hz
  unit  449 ( mua): r_lfp=-0.354, r_me=-0.398, mean_rate=255.07 Hz
  unit  446 ( mua): r_lfp=-0.351, r_me=-0.397, mean_rate=100.86 Hz
  unit  448 ( mua): r_lfp=-0.353, r_me=-0.397, mean_rate=217.71 Hz
  unit  4

In [7]:
# Screen for unit activity that leads later LFP/ME increases
RAMP_ANALYSIS_RATE_HZ = 20.0
RAMP_MAX_LEAD_SECONDS = 30.0
RAMP_LAG_STEP_SECONDS = 1.0
RAMP_MIN_LEAD_SECONDS = 1.0
RAMP_FUTURE_LEAD_SECONDS = 2.0
RAMP_FUTURE_WINDOW_SECONDS = 10.0
RAMP_TRACE_SMOOTH_SECONDS = 2.0
RAMP_ZERO_LAG_PENALTY = 1.0
RAMP_TOP_UNITS = 12

update_corr_trace_if_needed()
ramp_step = max(1, int(round(corr_trace.sample_rate_hz / RAMP_ANALYSIS_RATE_HZ)))
ramp_rate_hz = corr_trace.sample_rate_hz / ramp_step
ramp_spike_rate = decimate_spike_rate_result(spike_rate, ramp_step)
ramp_lfp_trace = corr_trace.lfp_trace[::ramp_step]
ramp_me_trace = corr_trace.motion_energy[::ramp_step]
ramp_valid_mask = spike_sorted_mask[::ramp_step]

lfp_lead = unit_trace_lag_correlations(
    ramp_spike_rate,
    ramp_lfp_trace,
    sample_rate_hz=ramp_rate_hz,
    max_lag_seconds=RAMP_MAX_LEAD_SECONDS,
    lag_step_seconds=RAMP_LAG_STEP_SECONDS,
    valid_mask=ramp_valid_mask,
    min_lead_seconds=RAMP_MIN_LEAD_SECONDS,
)
me_lead = unit_trace_lag_correlations(
    ramp_spike_rate,
    ramp_me_trace,
    sample_rate_hz=ramp_rate_hz,
    max_lag_seconds=RAMP_MAX_LEAD_SECONDS,
    lag_step_seconds=RAMP_LAG_STEP_SECONDS,
    valid_mask=ramp_valid_mask,
    min_lead_seconds=RAMP_MIN_LEAD_SECONDS,
)
lfp_future_change, lfp_future_valid = future_trace_change(
    ramp_lfp_trace,
    sample_rate_hz=ramp_rate_hz,
    lead_seconds=RAMP_FUTURE_LEAD_SECONDS,
    change_window_seconds=RAMP_FUTURE_WINDOW_SECONDS,
    smooth_seconds=RAMP_TRACE_SMOOTH_SECONDS,
    valid_mask=ramp_valid_mask,
)
me_future_change, me_future_valid = future_trace_change(
    ramp_me_trace,
    sample_rate_hz=ramp_rate_hz,
    lead_seconds=RAMP_FUTURE_LEAD_SECONDS,
    change_window_seconds=RAMP_FUTURE_WINDOW_SECONDS,
    smooth_seconds=RAMP_TRACE_SMOOTH_SECONDS,
    valid_mask=ramp_valid_mask,
)
lfp_future_r = correlate_unit_rates_to_trace(
    ramp_spike_rate,
    lfp_future_change,
    valid_mask=ramp_valid_mask & lfp_future_valid,
)
me_future_r = correlate_unit_rates_to_trace(
    ramp_spike_rate,
    me_future_change,
    valid_mask=ramp_valid_mask & me_future_valid,
)

lead_lfp_r = np.nan_to_num(lfp_lead.best_lead_r, nan=-np.inf)
lead_me_r = np.nan_to_num(me_lead.best_lead_r, nan=-np.inf)
future_lfp_r = np.nan_to_num(lfp_future_r, nan=-np.inf)
future_me_r = np.nan_to_num(me_future_r, nan=-np.inf)
zero_metric = np.maximum(
    np.nan_to_num(np.abs(spike_lfp_r), nan=0.0),
    np.nan_to_num(np.abs(spike_me_r), nan=0.0),
)
lead_metric = np.maximum(lead_lfp_r, lead_me_r)
future_metric = np.maximum(future_lfp_r, future_me_r)
ramp_score = np.maximum(lead_metric, future_metric) - RAMP_ZERO_LAG_PENALTY * zero_metric
ramp_ranked_units = np.argsort(ramp_score)[::-1]

print(
    f"Ramping/lead screen at {ramp_rate_hz:.1f} Hz: positive lead means unit activity "
    "comes before LFP/ME. Future-change score asks whether current firing predicts "
    f"a trace rise {RAMP_FUTURE_LEAD_SECONDS:g}-{RAMP_FUTURE_LEAD_SECONDS + RAMP_FUTURE_WINDOW_SECONDS:g}s later."
)
print("Top ramp/lead candidates:")
for rank_idx in ramp_ranked_units[:RAMP_TOP_UNITS]:
    unit_id = int(spike_rate.unit_ids[rank_idx])
    label = spike_rate.unit_labels[rank_idx] or "unlabeled"
    if lead_lfp_r[rank_idx] >= lead_me_r[rank_idx]:
        lead_source = "lfp"
        lead_s = lfp_lead.best_lead_s[rank_idx]
        lead_r = lfp_lead.best_lead_r[rank_idx]
    else:
        lead_source = "me"
        lead_s = me_lead.best_lead_s[rank_idx]
        lead_r = me_lead.best_lead_r[rank_idx]
    if future_lfp_r[rank_idx] >= future_me_r[rank_idx]:
        future_source = "lfp"
        future_r = lfp_future_r[rank_idx]
    else:
        future_source = "me"
        future_r = me_future_r[rank_idx]
    print(
        f"  unit {unit_id:>4d} ({label:>4s}): "
        f"score={ramp_score[rank_idx]:+.3f}, "
        f"best_lead={lead_r:+.3f} {lead_source} at {lead_s:.1f}s, "
        f"future_r={future_r:+.3f} {future_source}, "
        f"zero_lfp={spike_lfp_r[rank_idx]:+.3f}, zero_me={spike_me_r[rank_idx]:+.3f}"
    )


Ramping/lead screen at 20.8 Hz: positive lead means unit activity comes before LFP/ME. Future-change score asks whether current firing predicts a trace rise 2-12s later.
Top ramp/lead candidates:
  unit  413 (good): score=+0.046, best_lead=+0.108 lfp at 27.2s, future_r=+0.045 me, zero_lfp=+0.062, zero_me=-0.018
  unit  127 (good): score=+0.039, best_lead=+0.342 me at 1.0s, future_r=-0.075 me, zero_lfp=+0.303, zero_me=+0.296
  unit  512 ( mua): score=+0.034, best_lead=+0.070 lfp at 2.0s, future_r=+0.051 me, zero_lfp=+0.036, zero_me=-0.004
  unit  503 ( mua): score=+0.033, best_lead=+0.046 lfp at 2.0s, future_r=+0.028 me, zero_lfp=+0.013, zero_me=+0.005
  unit  513 ( mua): score=+0.033, best_lead=+0.059 lfp at 7.1s, future_r=+0.051 me, zero_lfp=+0.026, zero_me=-0.017
  unit  510 ( mua): score=+0.032, best_lead=+0.054 lfp at 7.1s, future_r=+0.049 me, zero_lfp=+0.018, zero_me=-0.021
  unit  509 ( mua): score=+0.031, best_lead=+0.053 lfp at 7.1s, future_r=+0.048 me, zero_lfp=+0.017, zero_me

In [27]:
# Plot unit spike rates aligned to LFP magnitude and motion energy
SPIKE_RATE_PLOT_ORDER = "correlation"      # "low_me_rate", "ramp", or "correlation"
SPIKE_RATE_LOW_ME_PERCENTILE = 10.0
SPIKE_RATE_PLOT_SMOOTH_SECONDS = .5
SPIKE_RATE_MAX_PLOT_UNITS = 50
SPIKE_RATE_MAX_PLOT_POINTS = 100_000

# Reuse the Kilosort/corr trace settings and load the processed trace cache if needed.
# This cell should not force a different filter or rebuild from raw unless the cache is missing/mismatched.
CORR_TRACE_CACHE = True
CORR_TRACE_CACHE_OVERWRITE = False
CORR_TRACE_CACHE_ALLOW_CHANNEL_MISMATCH = True
CORR_ME_ENVELOPE = globals().get("CORR_ME_ENVELOPE", globals().get("NO_UNIT_ME_ENVELOPE", False))
CORR_ME_ARTIFACT_FILTER = globals().get("CORR_ME_ARTIFACT_FILTER", globals().get("NO_UNIT_ME_ARTIFACT_FILTER", True))
if "corr_trace_settings" in globals():
    corr_trace_settings = dict(corr_trace_settings)
    corr_trace_settings["trace_cache_overwrite"] = False
update_corr_trace_if_needed()
spike_plot_start_min = 0.0
spike_plot_duration_min = 130.0
spike_plot_start_s = spike_plot_start_min * 60.0
spike_plot_stop_s = spike_plot_start_s + spike_plot_duration_min * 60.0
spike_plot_mask = (
    (corr_trace.t_video_s >= spike_plot_start_s)
    & (corr_trace.t_video_s < spike_plot_stop_s)
    & spike_sorted_mask
)
spike_plot_full_idx = np.flatnonzero(spike_plot_mask)
if spike_plot_full_idx.size == 0:
    raise ValueError("No Kilosort/LFP/ME overlap in requested spike-rate plot window")

if SPIKE_RATE_PLOT_ORDER == "low_me_rate":
    low_me_threshold = float(
        np.nanpercentile(corr_trace.motion_energy[spike_plot_full_idx], SPIKE_RATE_LOW_ME_PERCENTILE)
    )
    low_me_idx = spike_plot_full_idx[corr_trace.motion_energy[spike_plot_full_idx] <= low_me_threshold]
    if low_me_idx.size == 0:
        low_me_idx = spike_plot_full_idx
    low_me_mean_rate = np.nanmean(spike_rate.spike_rate_hz[:, low_me_idx], axis=1)
    unit_order_source = np.argsort(np.nan_to_num(low_me_mean_rate, nan=-np.inf))[::-1]
    order_detail = (
        f"low ME p{SPIKE_RATE_LOW_ME_PERCENTILE:g}, "
        f"n={low_me_idx.size:,}, threshold={low_me_threshold:.3g}"
    )
elif SPIKE_RATE_PLOT_ORDER == "ramp" and "ramp_ranked_units" in globals():
    unit_order_source = ramp_ranked_units
    order_detail = "ramp"
else:
    unit_order_source = ranked_units
    order_detail = "correlation"

spike_plot_step = max(1, int(np.ceil(spike_plot_full_idx.size / SPIKE_RATE_MAX_PLOT_POINTS)))
spike_plot_idx = spike_plot_full_idx[::spike_plot_step]
unit_plot_order = unit_order_source[: min(SPIKE_RATE_MAX_PLOT_UNITS, unit_order_source.size)]
rate_image = spike_rate.spike_rate_hz[unit_plot_order][:, spike_plot_idx]
if SPIKE_RATE_PLOT_SMOOTH_SECONDS > 0 and spike_plot_idx.size > 1:
    plot_rate_hz = 1.0 / float(np.nanmedian(np.diff(corr_trace.t_video_s[spike_plot_idx])))
    rate_image = gaussian_smooth_rows(rate_image, SPIKE_RATE_PLOT_SMOOTH_SECONDS * plot_rate_hz)
rate_image_z = np.vstack([zscore(row) for row in rate_image]).astype(np.float32, copy=False)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1.0, 1.0, 2.4]},
)
axes[0].plot(
    corr_trace.t_video_s[spike_plot_idx] / 60.0,
    zscore(corr_trace.lfp_trace[spike_plot_idx]),
    lw=0.9,
    label=corr_lfp_label,
)
axes[0].legend(loc="upper right")
axes[0].set_ylabel("LFP z")
axes[0].grid(alpha=0.25)

axes[1].plot(
    corr_trace.t_video_s[spike_plot_idx] / 60.0,
    zscore(corr_trace.motion_energy[spike_plot_idx]),
    lw=0.9,
    color="tab:orange",
    label=corr_me_label,
)
axes[1].legend(loc="upper right")
axes[1].set_ylabel("ME z")
axes[1].grid(alpha=0.25)

vlim = np.nanpercentile(np.abs(rate_image_z), 98) if np.isfinite(rate_image_z).any() else 1.0
im = axes[2].imshow(
    rate_image_z,
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=-vlim,
    vmax=vlim,
    extent=[
        corr_trace.t_video_s[spike_plot_idx[0]] / 60.0,
        corr_trace.t_video_s[spike_plot_idx[-1]] / 60.0,
        len(unit_plot_order) - 0.5,
        -0.5,
    ],
)
axes[2].set_yticks(np.arange(len(unit_plot_order)))
axes[2].set_yticklabels([str(int(spike_rate.unit_ids[idx])) for idx in unit_plot_order])
axes[2].set_ylabel("unit id")
axes[2].set_xlabel("video time (min)")
axes[2].set_title(
    "Kilosort unit spike rates, "
    f"order={order_detail}, plot smoothing={SPIKE_RATE_PLOT_SMOOTH_SECONDS:g}s"
)
fig.colorbar(im, ax=axes[2], label="spike-rate z")
plt.show()


In [9]:
# Whole-dataset channel-average correlation with block-local filter parameters
FULL_CORR_CHANNEL_START = 50
FULL_CORR_CHANNEL_STOP = 51
FULL_CORR_CHANNEL_STEP = 1
FULL_CORR_CHUNK_SECONDS = 360.0
FULL_CORR_REDUCTION = "mean"     # "mean" or "mean_abs"
FULL_CORR_LFP_HIGHPASS_HZ = None
FULL_CORR_LFP_LOWPASS_HZ = 5.0
FULL_CORR_ME_HIGHPASS_HZ = None
FULL_CORR_ME_LOWPASS_HZ = 5.0
FULL_CORR_FILTER_ORDER = 3
FULL_CORR_MAX_PLOT_POINTS = 100_000
FULL_CORR_MAX_SCATTER_POINTS = 75_000
TARGET_RATE_HZ = 250.0
LFP_READ_CHUNK_SECONDS = 1.0
LFP_READ_MODE = "auto"

full_channels = channel_range(FULL_CORR_CHANNEL_START, FULL_CORR_CHANNEL_STOP, FULL_CORR_CHANNEL_STEP)
importlib.reload(lme_utils)
full_t, full_lfp_mean, full_sample_rate_hz = lme_utils.load_channel_average_lfp(
    recording,
    full_channels,
    chunk_seconds=FULL_CORR_CHUNK_SECONDS,
    target_rate_hz=TARGET_RATE_HZ,
    read_chunk_seconds=LFP_READ_CHUNK_SECONDS,
    read_mode=LFP_READ_MODE,
    reduction=FULL_CORR_REDUCTION,
)
full_me_cache_path = lme_utils.find_lfp_motion_energy_cache(
    recording,
    full_channels,
    target_rate_hz=TARGET_RATE_HZ,
)
if full_me_cache_path is not None:
    full_me_cache = lme_utils.load_lfp_motion_energy_cache(
        full_me_cache_path,
        channels=full_channels,
        load_lfp=False,
    )
    full_me = full_me_cache.motion_energy
else:
    full_me = recording.sample_motion_energy(full_t)
full_lfp_filtered = butter_filter_1d(
    full_lfp_mean,
    sample_rate_hz=full_sample_rate_hz,
    highpass_hz=FULL_CORR_LFP_HIGHPASS_HZ,
    lowpass_hz=FULL_CORR_LFP_LOWPASS_HZ,
    order=FULL_CORR_FILTER_ORDER,
)
full_me_filtered = butter_filter_1d(
    full_me,
    sample_rate_hz=full_sample_rate_hz,
    highpass_hz=FULL_CORR_ME_HIGHPASS_HZ,
    lowpass_hz=FULL_CORR_ME_LOWPASS_HZ,
    order=FULL_CORR_FILTER_ORDER,
)
full_lfp_envelope = amplitude_envelope_1d(full_lfp_filtered)
full_me_envelope = amplitude_envelope_1d(full_me_filtered)

full_r, full_valid = pearsonr_valid(full_lfp_envelope, full_me_envelope)
print(
    f"Whole-dataset filtered-envelope Pearson r = {full_r:+.4f} over {full_valid.sum():,} samples "
    f"at {full_sample_rate_hz:.1f} Hz; "
    f"channels {full_channels[0]}-{full_channels[-1]}, reduction={FULL_CORR_REDUCTION}"
)

full_plot_step = max(1, int(np.ceil(len(full_t) / FULL_CORR_MAX_PLOT_POINTS)))
full_scatter_step = max(1, int(np.ceil(full_valid.sum() / FULL_CORR_MAX_SCATTER_POINTS)))
full_scatter_idx = np.flatnonzero(full_valid)[::full_scatter_step]

fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
axes[0].plot(
    full_t[::full_plot_step] / 60.0,
    zscore(full_lfp_envelope)[::full_plot_step],
    label="filtered LFP envelope",
    lw=0.9,
)
axes[0].plot(
    full_t[::full_plot_step] / 60.0,
    zscore(full_me_envelope)[::full_plot_step],
    label="filtered ME envelope",
    lw=0.9,
)
axes[0].set_title(
    f"Whole-dataset filtered-envelope correlation at {full_sample_rate_hz:.1f} Hz, "
    f"LF channels {full_channels[0]}-{full_channels[-1]} (r={full_r:+.3f})"
)
axes[0].set_xlabel("video time (min)")
axes[0].set_ylabel("z-score")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].scatter(
    full_me_envelope[full_scatter_idx],
    full_lfp_envelope[full_scatter_idx],
    s=4,
    alpha=0.18,
)
axes[1].set_title("Envelope samples, downsampled for display")
axes[1].set_xlabel("filtered motion-energy envelope")
axes[1].set_ylabel("filtered channel-average LFP envelope (uV)")
axes[1].grid(alpha=0.25)
plt.show()


using cached LFP data: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
Whole-dataset filtered-envelope Pearson r = +0.2463 over 1,984,710 samples at 250.0 Hz; channels 50-50, reduction=mean


In [ ]:
# Diagnostic block: compare low-pass choices for ME and LFP envelopes
DIAG_LOWPASS_HZ = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
DIAG_TRACE_LOWPASS_HZ = 1.0
DIAG_LAG_SECONDS = 30.0
DIAG_LAG_RATE_HZ = 20.0

diagnostic_rows = []
for cutoff_hz in DIAG_LOWPASS_HZ:
    lfp_f = butter_filter_1d(
        corr_trace.lfp_trace,
        sample_rate_hz=corr_trace.sample_rate_hz,
        lowpass_hz=cutoff_hz,
        order=3,
    )
    me_f = butter_filter_1d(
        corr_trace.motion_energy,
        sample_rate_hz=corr_trace.sample_rate_hz,
        lowpass_hz=cutoff_hz,
        order=3,
    )
    r0, valid_f = pearsonr_valid(lfp_f, me_f)
    me_lag, lag_rate_hz = decimate_for_lag(zscore(me_f), corr_trace.sample_rate_hz, DIAG_LAG_RATE_HZ)
    lfp_lag, _lag_rate_hz = decimate_for_lag(zscore(lfp_f), corr_trace.sample_rate_hz, DIAG_LAG_RATE_HZ)
    lags_s, lag_r = lagged_pearson(
        me_lag,
        lfp_lag,
        sample_rate_hz=lag_rate_hz,
        max_lag_seconds=DIAG_LAG_SECONDS,
    )
    best_idx = int(np.nanargmax(np.abs(lag_r))) if np.isfinite(lag_r).any() else 0
    diagnostic_rows.append(
        {
            "cutoff_hz": float(cutoff_hz),
            "r_zero_lag": float(r0),
            "best_lag_s": float(lags_s[best_idx]),
            "best_lag_r": float(lag_r[best_idx]),
            "lfp_filtered": lfp_f,
            "me_filtered": me_f,
            "lags_s": lags_s,
            "lag_r": lag_r,
        }
    )

print("Low-pass diagnostic:")
for row in diagnostic_rows:
    print(
        f"  <= {row['cutoff_hz']:>4g} Hz: "
        f"r0={row['r_zero_lag']:+.3f}, "
        f"best |r|={row['best_lag_r']:+.3f} at lag={row['best_lag_s']:+.2f}s"
    )

trace_row = min(diagnostic_rows, key=lambda row: abs(row["cutoff_hz"] - float(DIAG_TRACE_LOWPASS_HZ)))
plot_step = max(1, int(np.ceil(len(corr_trace.t_video_s) / 100_000)))
valid_trace = np.isfinite(trace_row["lfp_filtered"]) & np.isfinite(trace_row["me_filtered"])
scatter_step = max(1, int(np.ceil(valid_trace.sum() / 75_000)))
scatter_idx = np.flatnonzero(valid_trace)[::scatter_step]

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
axes[0, 0].plot(
    corr_trace.t_video_s[::plot_step] / 60.0,
    zscore(trace_row["lfp_filtered"])[::plot_step],
    label="low-pass LFP envelope",
    lw=0.9,
)
axes[0, 0].plot(
    corr_trace.t_video_s[::plot_step] / 60.0,
    zscore(trace_row["me_filtered"])[::plot_step],
    label="low-pass ME envelope",
    lw=0.9,
)
axes[0, 0].set_title(f"Low-pass <= {trace_row['cutoff_hz']:g} Hz")
axes[0, 0].set_xlabel("video time (min)")
axes[0, 0].set_ylabel("z-score")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].scatter(
    trace_row["me_filtered"][scatter_idx],
    trace_row["lfp_filtered"][scatter_idx],
    s=4,
    alpha=0.18,
)
axes[0, 1].set_title(f"Scatter, r0={trace_row['r_zero_lag']:+.3f}")
axes[0, 1].set_xlabel("low-pass ME envelope")
axes[0, 1].set_ylabel("low-pass LFP envelope (uV)")
axes[0, 1].grid(alpha=0.25)

cutoffs = np.asarray([row["cutoff_hz"] for row in diagnostic_rows], dtype=float)
r0 = np.asarray([row["r_zero_lag"] for row in diagnostic_rows], dtype=float)
best_r = np.asarray([row["best_lag_r"] for row in diagnostic_rows], dtype=float)
axes[1, 0].semilogx(cutoffs, r0, marker="o", label="zero lag")
axes[1, 0].semilogx(cutoffs, best_r, marker="o", label=f"best lag within +/-{DIAG_LAG_SECONDS:g}s")
axes[1, 0].axhline(0, color="0.5", lw=0.8)
axes[1, 0].set_title("Correlation vs low-pass cutoff")
axes[1, 0].set_xlabel("low-pass cutoff (Hz)")
axes[1, 0].set_ylabel("Pearson r")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25, which="both")

axes[1, 1].plot(trace_row["lags_s"], trace_row["lag_r"], color="tab:purple")
axes[1, 1].axvline(0, color="0.5", lw=0.8)
axes[1, 1].axhline(0, color="0.5", lw=0.8)
axes[1, 1].set_title(f"Lag correlation, <= {trace_row['cutoff_hz']:g} Hz")
axes[1, 1].set_xlabel("lag (s), positive = ME leads LFP")
axes[1, 1].set_ylabel("Pearson r")
axes[1, 1].grid(alpha=0.25)
plt.show()


In [16]:
# LFP frequency-content window loader
FREQ_LFP_START_MIN = 26.2
FREQ_LFP_DURATION_S = 1200.0  # None uses the full LFP/video overlap after start
FREQ_LFP_TARGET_RATE_HZ = 250.0
FREQ_LFP_CHANNELS = None     # set to [50, 150, 280] to override the range below
FREQ_LFP_CHANNEL_START = 50
FREQ_LFP_CHANNEL_STOP = 51
FREQ_LFP_CHANNEL_STEP = 1
FREQ_LFP_USE_CACHE = True
FREQ_LFP_READ_CHUNK_SECONDS = 1.0
FREQ_LFP_READ_MODE = "auto"
FREQ_SUBTRACT_CHANNEL_MEAN = True
FREQ_GLOBAL_MEDIAN_REMOVE = True
FREQ_LFP_HIGHPASS_HZ = None
FREQ_LFP_LOWPASS_HZ = None
FREQ_FILTER_ORDER = 3


def resolve_freq_channels(channels=None):
    if channels is None:
        stop = recording.lfp.n_lfp_channels if FREQ_LFP_CHANNEL_STOP is None else FREQ_LFP_CHANNEL_STOP
        channels = channel_range(FREQ_LFP_CHANNEL_START, stop, FREQ_LFP_CHANNEL_STEP)
    return lme_utils.validate_lfp_channels(np.asarray(channels, dtype=np.int64), recording.lfp.n_lfp_channels)


def preprocess_lfp_for_frequency(lfp_uv, sample_rate_hz):
    lfp = lfp_uv.astype(np.float32, copy=True)
    if FREQ_SUBTRACT_CHANNEL_MEAN:
        lfp = subtract_static_channel_mean(lfp)
    if FREQ_GLOBAL_MEDIAN_REMOVE:
        if lfp.shape[0] > 1:
            lfp = lfp - np.nanmedian(lfp, axis=0, keepdims=True).astype(np.float32)
        else:
            print("global median removal skipped for a single loaded channel")
    if FREQ_LFP_HIGHPASS_HZ is not None or FREQ_LFP_LOWPASS_HZ is not None:
        for row in range(lfp.shape[0]):
            lfp[row] = butter_filter_1d(
                lfp[row],
                sample_rate_hz=sample_rate_hz,
                highpass_hz=FREQ_LFP_HIGHPASS_HZ,
                lowpass_hz=FREQ_LFP_LOWPASS_HZ,
                order=FREQ_FILTER_ORDER,
            )
    return lfp


def load_frequency_lfp_window(channels=None, *, start_min=None, duration_s=None):
    channels = resolve_freq_channels(channels)
    start_min = FREQ_LFP_START_MIN if start_min is None else float(start_min)
    duration_s = FREQ_LFP_DURATION_S if duration_s is None else duration_s
    overlap_start_s, overlap_stop_s = recording.lfp_video_overlap()
    start_s = max(float(start_min) * 60.0, overlap_start_s)
    stop_s = overlap_stop_s if duration_s is None else min(start_s + float(duration_s), overlap_stop_s)
    if stop_s <= start_s:
        raise ValueError(f"No LFP/video overlap in requested frequency window: {start_s:.3f}-{stop_s:.3f}s")

    cache_path = None
    if FREQ_LFP_USE_CACHE:
        cache_path = lme_utils.find_lfp_motion_energy_cache(
            recording,
            channels,
            target_rate_hz=FREQ_LFP_TARGET_RATE_HZ,
            start_s=start_s,
            stop_s=stop_s,
        )
    if cache_path is not None:
        print(f"using cached LFP frequency data: {cache_path}")
        loaded = lme_utils.load_lfp_motion_energy_cache(
            cache_path,
            start_s=start_s,
            duration_s=stop_s - start_s,
            channels=channels,
            load_lfp=True,
        )
        t_video_s = loaded.t_video_s
        lfp_uv = loaded.lfp_uv
        sample_rate_hz = loaded.sample_rate_hz
    else:
        print("LFP frequency cache not found; reading raw LF chunks")
        loaded = recording.load_lfp_downsampled(
            start_s,
            stop_s - start_s,
            channels,
            target_rate_hz=FREQ_LFP_TARGET_RATE_HZ,
            read_chunk_seconds=FREQ_LFP_READ_CHUNK_SECONDS,
            read_mode=FREQ_LFP_READ_MODE,
        )
        t_video_s = loaded.t_video_s
        lfp_uv = loaded.lfp_uv
        sample_rate_hz = loaded.sample_rate_hz

    if t_video_s.size == 0:
        raise ValueError("Selected LFP frequency window is empty")
    lfp_processed = preprocess_lfp_for_frequency(lfp_uv, sample_rate_hz)
    print(
        f"loaded frequency LFP: {len(channels)} channel(s) x {t_video_s.size:,} samples, "
        f"{sample_rate_hz:.2f} Hz, video time {t_video_s[0] / 60:.2f}-{t_video_s[-1] / 60:.2f} min"
    )
    return channels, t_video_s, lfp_processed, sample_rate_hz


freq_lfp_channels, freq_lfp_t_video_s, freq_lfp_uv, freq_lfp_sample_rate_hz = load_frequency_lfp_window(FREQ_LFP_CHANNELS)



using cached LFP frequency data: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
global median removal skipped for a single loaded channel
loaded frequency LFP: 1 channel(s) x 300,000 samples, 250.00 Hz, video time 26.20-46.20 min


In [17]:
# Welch power spectra for the selected LFP window
try:
    import ephys_functions as ef
    calculate_psd = ef.calculate_power_spectral_density
except Exception as exc:
    print(f"Could not import ephys_functions.calculate_power_spectral_density ({type(exc).__name__}: {exc}); using scipy.signal.welch")
    from scipy.signal import welch

    def calculate_psd(signal, sampling_rate, nperseg=1024, noverlap=None, window="hann", detrend="constant", nfft=None):
        return welch(
            signal,
            fs=sampling_rate,
            window=window,
            nperseg=nperseg,
            noverlap=noverlap,
            detrend=detrend,
            nfft=nfft,
            scaling="density",
            average="mean",
        )

FREQ_PSD_MIN_HZ = 0.5
FREQ_PSD_MAX_HZ = 100.0
FREQ_PSD_WINDOW_SECONDS = 8.0
FREQ_PSD_OVERLAP_FRACTION = 0.5
FREQ_PSD_NFFT = None
FREQ_PSD_MAX_CHANNEL_LINES = 60


def fill_nans_for_spectrum(values):
    values = np.asarray(values, dtype=np.float64).copy()
    finite = np.isfinite(values)
    if finite.sum() == 0:
        raise ValueError("Cannot compute spectrum: trace has no finite samples")
    if finite.sum() < values.size:
        idx = np.arange(values.size, dtype=np.float64)
        values[~finite] = np.interp(idx[~finite], idx[finite], values[finite])
    values -= np.nanmean(values)
    return values

nperseg = int(round(FREQ_PSD_WINDOW_SECONDS * freq_lfp_sample_rate_hz))
nperseg = max(8, min(nperseg, freq_lfp_uv.shape[1]))
noverlap = int(round(FREQ_PSD_OVERLAP_FRACTION * nperseg))
noverlap = min(max(0, noverlap), nperseg - 1)

psd_rows = []
psd_freqs = None
for trace in freq_lfp_uv:
    freqs, psd = calculate_psd(
        fill_nans_for_spectrum(trace),
        freq_lfp_sample_rate_hz,
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=FREQ_PSD_NFFT,
    )
    if psd_freqs is None:
        psd_freqs = freqs
    psd_rows.append(psd)
psd_matrix = np.asarray(psd_rows, dtype=np.float64)
psd_mask = (psd_freqs >= FREQ_PSD_MIN_HZ) & (psd_freqs <= min(FREQ_PSD_MAX_HZ, freq_lfp_sample_rate_hz / 2.0))
if not psd_mask.any():
    raise ValueError("No PSD frequency bins in requested frequency range")
median_psd = np.nanmedian(psd_matrix, axis=0)
mean_trace = np.nanmean(freq_lfp_uv, axis=0)
mean_freqs, mean_psd = calculate_psd(
    fill_nans_for_spectrum(mean_trace),
    freq_lfp_sample_rate_hz,
    nperseg=nperseg,
    noverlap=noverlap,
    nfft=FREQ_PSD_NFFT,
)

line_step = max(1, int(np.ceil(freq_lfp_channels.size / FREQ_PSD_MAX_CHANNEL_LINES)))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for channel, psd in zip(freq_lfp_channels[::line_step], psd_matrix[::line_step]):
    axes[0].loglog(psd_freqs[psd_mask], psd[psd_mask], lw=0.7, alpha=0.25, color="tab:blue")
axes[0].loglog(psd_freqs[psd_mask], median_psd[psd_mask], lw=2.0, color="black", label="median channel PSD")
axes[0].loglog(mean_freqs[psd_mask], mean_psd[psd_mask], lw=1.5, color="tab:red", label="mean trace PSD")
axes[0].set_title(
    f"Welch PSD, {freq_lfp_t_video_s[0] / 60:.2f}-{freq_lfp_t_video_s[-1] / 60:.2f} min, "
    f"nperseg={nperseg:,}"
)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("PSD (uV^2/Hz)")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.25, which="both")

psd_db = 10.0 * np.log10(psd_matrix[:, psd_mask] + np.finfo(float).tiny)
lo, hi = np.nanpercentile(psd_db, [5, 99])
pcm = axes[1].pcolormesh(
    psd_freqs[psd_mask],
    freq_lfp_channels,
    psd_db,
    shading="auto",
    cmap="magma",
    vmin=lo,
    vmax=hi,
)
axes[1].set_xscale("log")
axes[1].set_title("Channel PSD heatmap")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("LFP channel")
fig.colorbar(pcm, ax=axes[1], label="PSD (dB uV^2/Hz)")
plt.show()



In [26]:
# Movement vs stationary LFP power spectra
# Uses the already loaded/preprocessed frequency LFP window from the frequency loader block.
MOVEMENT_PSD_ME_Z_THRESHOLD = -0.3
MOVEMENT_PSD_ME_LOWPASS_HZ = 0.5
MOVEMENT_PSD_ME_LOWPASS_ORDER = 3
MOVEMENT_PSD_ME_PLOT_MAX_POINTS = 20000
MOVEMENT_PSD_MIN_HZ = FREQ_PSD_MIN_HZ if "FREQ_PSD_MIN_HZ" in globals() else 0.5
MOVEMENT_PSD_MAX_HZ = FREQ_PSD_MAX_HZ if "FREQ_PSD_MAX_HZ" in globals() else 100.0
MOVEMENT_PSD_WINDOW_SECONDS = FREQ_PSD_WINDOW_SECONDS if "FREQ_PSD_WINDOW_SECONDS" in globals() else 8.0
MOVEMENT_PSD_OVERLAP_FRACTION = FREQ_PSD_OVERLAP_FRACTION if "FREQ_PSD_OVERLAP_FRACTION" in globals() else 0.5
MOVEMENT_PSD_NFFT = FREQ_PSD_NFFT if "FREQ_PSD_NFFT" in globals() else None
MOVEMENT_PSD_MIN_SEGMENT_SECONDS = MOVEMENT_PSD_WINDOW_SECONDS
MOVEMENT_PSD_MAX_CHANNEL_LINES = 60
MOVEMENT_PSD_USE_CORR_TRACE_ME = True
MOVEMENT_PSD_CLOSE_OLD_FIGURES = False

if "freq_lfp_uv" not in globals() or "freq_lfp_t_video_s" not in globals():
    if "load_frequency_lfp_window" not in globals():
        raise RuntimeError("Run the LFP frequency-content window loader block first")
    freq_lfp_channels, freq_lfp_t_video_s, freq_lfp_uv, freq_lfp_sample_rate_hz = load_frequency_lfp_window(FREQ_LFP_CHANNELS)

if "calculate_psd" not in globals():
    from scipy.signal import welch

    def calculate_psd(signal, sampling_rate, nperseg=1024, noverlap=None, window="hann", detrend="constant", nfft=None):
        return welch(
            signal,
            fs=sampling_rate,
            window=window,
            nperseg=nperseg,
            noverlap=noverlap,
            detrend=detrend,
            nfft=nfft,
            scaling="density",
            average="mean",
        )

if "fill_nans_for_spectrum" not in globals():
    def fill_nans_for_spectrum(values):
        values = np.asarray(values, dtype=np.float64).copy()
        finite = np.isfinite(values)
        if finite.sum() == 0:
            raise ValueError("Cannot compute spectrum: trace has no finite samples")
        if finite.sum() < values.size:
            idx = np.arange(values.size, dtype=np.float64)
            values[~finite] = np.interp(idx[~finite], idx[finite], values[finite])
        values -= np.nanmean(values)
        return values


def lowpass_trace_for_state_mask(values, sample_rate_hz, cutoff_hz, order=3):
    values = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(values)
    if finite.sum() == 0:
        raise ValueError("Cannot low-pass motion energy: trace has no finite samples")
    filled = values.copy()
    if finite.sum() < values.size:
        idx = np.arange(values.size, dtype=np.float64)
        filled[~finite] = np.interp(idx[~finite], idx[finite], values[finite])
    nyquist_hz = 0.5 * float(sample_rate_hz)
    if cutoff_hz is None or cutoff_hz <= 0 or cutoff_hz >= nyquist_hz:
        filtered = filled.copy()
    else:
        from scipy.signal import butter, sosfilt, sosfiltfilt

        sos = butter(order, cutoff_hz, btype="lowpass", fs=sample_rate_hz, output="sos")
        try:
            filtered = sosfiltfilt(sos, filled)
        except ValueError:
            filtered = sosfilt(sos, filled)
    filtered[~finite] = np.nan
    return filtered


def contiguous_true_runs(mask, min_len=1):
    mask = np.asarray(mask, dtype=bool)
    edges = np.diff(mask.astype(np.int8), prepend=0, append=0)
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    return [(int(s), int(e)) for s, e in zip(starts, stops) if (e - s) >= int(min_len)]


def state_psd_by_channel(lfp_uv, sample_rate_hz, runs, *, nperseg, noverlap, nfft=None):
    if not runs:
        raise ValueError("No runs are long enough for PSD calculation")
    psd_rows = []
    freqs = None
    run_lengths = np.asarray([stop - start for start, stop in runs], dtype=np.float64)
    for trace in lfp_uv:
        segment_psds = []
        segment_weights = []
        for (start, stop), weight in zip(runs, run_lengths):
            segment = fill_nans_for_spectrum(trace[start:stop])
            if segment.size < nperseg:
                continue
            f, pxx = calculate_psd(
                segment,
                sample_rate_hz,
                nperseg=nperseg,
                noverlap=min(noverlap, nperseg - 1),
                nfft=nfft,
            )
            if freqs is None:
                freqs = f
            segment_psds.append(pxx)
            segment_weights.append(weight)
        if not segment_psds:
            psd_rows.append(np.full_like(freqs, np.nan, dtype=np.float64) if freqs is not None else np.empty(0))
            continue
        psd_rows.append(np.average(np.asarray(segment_psds), axis=0, weights=np.asarray(segment_weights)))
    return freqs, np.asarray(psd_rows, dtype=np.float64)

if MOVEMENT_PSD_USE_CORR_TRACE_ME and "corr_trace" in globals():
    movement_me = np.interp(
        freq_lfp_t_video_s,
        corr_trace.t_video_s,
        corr_trace.motion_energy.astype(np.float64, copy=False),
        left=np.nan,
        right=np.nan,
    )
    movement_me_source = "corr_trace.motion_energy"
else:
    movement_me = recording.sample_motion_energy(freq_lfp_t_video_s)
    movement_me_source = "recording.motion_energy"

movement_me_raw_z = zscore(movement_me)
movement_me_lowpass = lowpass_trace_for_state_mask(
    movement_me,
    freq_lfp_sample_rate_hz,
    MOVEMENT_PSD_ME_LOWPASS_HZ,
    order=MOVEMENT_PSD_ME_LOWPASS_ORDER,
)
movement_me_z = zscore(movement_me_lowpass)
movement_mask = movement_me_z > MOVEMENT_PSD_ME_Z_THRESHOLD
stationary_mask = movement_me_z <= MOVEMENT_PSD_ME_Z_THRESHOLD
finite_mask = np.isfinite(movement_me_z)
movement_mask &= finite_mask
stationary_mask &= finite_mask

movement_nperseg = int(round(MOVEMENT_PSD_WINDOW_SECONDS * freq_lfp_sample_rate_hz))
movement_nperseg = max(8, min(movement_nperseg, freq_lfp_uv.shape[1]))
movement_noverlap = int(round(MOVEMENT_PSD_OVERLAP_FRACTION * movement_nperseg))
movement_noverlap = min(max(0, movement_noverlap), movement_nperseg - 1)
min_run_samples = max(1, int(round(MOVEMENT_PSD_MIN_SEGMENT_SECONDS * freq_lfp_sample_rate_hz)))

movement_runs = contiguous_true_runs(movement_mask, min_run_samples)
stationary_runs = contiguous_true_runs(stationary_mask, min_run_samples)
if not movement_runs:
    raise ValueError(
        f"No movement runs >= {MOVEMENT_PSD_MIN_SEGMENT_SECONDS:g}s with low-pass ME z > {MOVEMENT_PSD_ME_Z_THRESHOLD:g}"
    )
if not stationary_runs:
    raise ValueError(
        f"No stationary runs >= {MOVEMENT_PSD_MIN_SEGMENT_SECONDS:g}s with low-pass ME z <= {MOVEMENT_PSD_ME_Z_THRESHOLD:g}"
    )

movement_freqs, movement_psd = state_psd_by_channel(
    freq_lfp_uv,
    freq_lfp_sample_rate_hz,
    movement_runs,
    nperseg=movement_nperseg,
    noverlap=movement_noverlap,
    nfft=MOVEMENT_PSD_NFFT,
)
stationary_freqs, stationary_psd = state_psd_by_channel(
    freq_lfp_uv,
    freq_lfp_sample_rate_hz,
    stationary_runs,
    nperseg=movement_nperseg,
    noverlap=movement_noverlap,
    nfft=MOVEMENT_PSD_NFFT,
)
if not np.allclose(movement_freqs, stationary_freqs):
    raise ValueError("Movement and stationary PSD frequency axes differ")

state_freq_mask = (
    (movement_freqs >= MOVEMENT_PSD_MIN_HZ)
    & (movement_freqs <= min(MOVEMENT_PSD_MAX_HZ, freq_lfp_sample_rate_hz / 2.0))
)
if not state_freq_mask.any():
    raise ValueError("No PSD frequency bins in requested movement/stationary frequency range")

movement_median = np.nanmedian(movement_psd, axis=0)
stationary_median = np.nanmedian(stationary_psd, axis=0)
movement_q25, movement_q75 = np.nanpercentile(movement_psd, [25, 75], axis=0)
stationary_q25, stationary_q75 = np.nanpercentile(stationary_psd, [25, 75], axis=0)
psd_ratio_db = 10.0 * np.log10((movement_psd + np.finfo(float).tiny) / (stationary_psd + np.finfo(float).tiny))
median_ratio_db = np.nanmedian(psd_ratio_db, axis=0)

movement_seconds = float(np.count_nonzero(movement_mask)) / freq_lfp_sample_rate_hz
stationary_seconds = float(np.count_nonzero(stationary_mask)) / freq_lfp_sample_rate_hz
print(
    f"ME source={movement_me_source}; {MOVEMENT_PSD_ME_LOWPASS_HZ:g} Hz low-pass movement z>{MOVEMENT_PSD_ME_Z_THRESHOLD:g}: "
    f"{movement_seconds / 60.0:.2f} min in {len(movement_runs)} run(s); "
    f"stationary: {stationary_seconds / 60.0:.2f} min in {len(stationary_runs)} run(s)"
)

if MOVEMENT_PSD_CLOSE_OLD_FIGURES:
    plt.close("all")
plt.ion()
me_plot_step = max(1, int(np.ceil(freq_lfp_t_video_s.size / MOVEMENT_PSD_ME_PLOT_MAX_POINTS)))
me_plot_t = freq_lfp_t_video_s[::me_plot_step]
fig_me, ax_me = plt.subplots(1, 1, figsize=(13, 2.8), constrained_layout=True)
ax_me.plot(me_plot_t, movement_me_raw_z[::me_plot_step], color="0.65", lw=0.7, alpha=0.55, label="raw ME z")
ax_me.plot(
    me_plot_t,
    movement_me_z[::me_plot_step],
    color="black",
    lw=1.2,
    label=f"{MOVEMENT_PSD_ME_LOWPASS_HZ:g} Hz low-pass ME z",
)
ax_me.axhline(MOVEMENT_PSD_ME_Z_THRESHOLD, color="tab:red", lw=1.0, ls="--", label="movement threshold")
me_y0, me_y1 = ax_me.get_ylim()
ax_me.fill_between(
    me_plot_t,
    me_y0,
    me_y1,
    where=movement_mask[::me_plot_step],
    color="tab:red",
    alpha=0.10,
    linewidth=0,
    label="movement mask",
)
ax_me.set_ylim(me_y0, me_y1)
ax_me.set_xlabel("Video time (s)")
ax_me.set_ylabel("ME z")
ax_me.set_title(f"Motion-energy state mask from {MOVEMENT_PSD_ME_LOWPASS_HZ:g} Hz low-pass ME")
ax_me.legend(frameon=False, ncol=4)
ax_me.grid(alpha=0.25)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
f = movement_freqs[state_freq_mask]
axes[0].loglog(f, movement_median[state_freq_mask], color="tab:red", lw=2.0, label=f"movement low-pass ME z>{MOVEMENT_PSD_ME_Z_THRESHOLD:g}")
axes[0].fill_between(f, movement_q25[state_freq_mask], movement_q75[state_freq_mask], color="tab:red", alpha=0.18, linewidth=0)
axes[0].loglog(f, stationary_median[state_freq_mask], color="tab:blue", lw=2.0, label=f"stationary low-pass ME z<={MOVEMENT_PSD_ME_Z_THRESHOLD:g}")
axes[0].fill_between(f, stationary_q25[state_freq_mask], stationary_q75[state_freq_mask], color="tab:blue", alpha=0.18, linewidth=0)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("PSD (uV^2/Hz)")
axes[0].set_title(
    f"LFP PSD by movement state, channels {lme_utils.channel_label(freq_lfp_channels)}, nperseg={movement_nperseg:,}"
)
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.25, which="both")

axes[1].semilogx(f, median_ratio_db[state_freq_mask], color="black", lw=2.0, label="median channel")
axes[1].axhline(0, color="0.5", lw=0.8)
line_step = max(1, int(np.ceil(freq_lfp_channels.size / MOVEMENT_PSD_MAX_CHANNEL_LINES)))
for channel, ratio in zip(freq_lfp_channels[::line_step], psd_ratio_db[::line_step]):
    axes[1].semilogx(f, ratio[state_freq_mask], color="0.2", lw=0.6, alpha=0.18)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Movement/stationary PSD (dB)")
axes[1].set_title("Power difference")
axes[1].grid(alpha=0.25, which="both")
plt.show(block=False)


ME source=corr_trace.motion_energy; 0.5 Hz low-pass movement z>-0.3: 9.79 min in 20 run(s); stationary: 10.21 min in 10 run(s)


In [21]:
# Wavelet transform plot for a selected LFP channel and time range
# This follows ephys_functions.plot_cwt_spectrogram: complex Morlet, log-spaced scales/frequencies, log y-axis.
WAVELET_CHANNEL = 50
WAVELET_START_MIN = 25
WAVELET_DURATION_S = 300.0
WAVELET_MIN_FREQ_HZ = 1.0
WAVELET_MAX_FREQ_HZ = 100.0
WAVELET_N_FREQS = 100
WAVELET_NAME = "cmor1.5-1.0"
WAVELET_FALLBACK_N_CYCLES = 6.0
WAVELET_POWER_SMOOTH_SIGMA = 1.0
WAVELET_VMIN_PERCENTILE = 5.0
WAVELET_VMAX_PERCENTILE = 99.0

from scipy.ndimage import gaussian_filter
from scipy.signal import fftconvolve


def cwt_power_like_ephys_functions(values, sample_rate_hz):
    values = fill_nans_for_spectrum(values)
    min_freq = float(WAVELET_MIN_FREQ_HZ)
    max_freq = min(float(WAVELET_MAX_FREQ_HZ), float(sample_rate_hz) / 2.0 * 0.98)
    if max_freq <= min_freq:
        raise ValueError(f"Invalid wavelet frequency range {min_freq:g}-{max_freq:g} Hz for fs={sample_rate_hz:g} Hz")

    try:
        import pywt
        sampling_period = 1.0 / float(sample_rate_hz)
        min_scale = 1.0 / max_freq * float(sample_rate_hz)
        max_scale = 1.0 / min_freq * float(sample_rate_hz)
        widths = np.geomspace(min_scale, max_scale, num=int(WAVELET_N_FREQS))
        cwt_matrix, freqs = pywt.cwt(values, widths, WAVELET_NAME, sampling_period=sampling_period)
        method = f"PyWavelets {WAVELET_NAME}"
    except ModuleNotFoundError:
        freqs = np.geomspace(min_freq, max_freq, num=int(WAVELET_N_FREQS))
        cwt_matrix = np.empty((freqs.size, values.size), dtype=np.complex64)
        for row, freq_hz in enumerate(freqs):
            sigma_t = float(WAVELET_FALLBACK_N_CYCLES) / (2.0 * np.pi * float(freq_hz))
            half_width_s = max(4.0 * sigma_t, 1.5 / float(freq_hz))
            half_width_samples = max(4, int(np.ceil(half_width_s * sample_rate_hz)))
            t_wave = np.arange(-half_width_samples, half_width_samples + 1, dtype=np.float64) / sample_rate_hz
            wavelet = np.exp(2j * np.pi * float(freq_hz) * t_wave) * np.exp(-(t_wave**2) / (2.0 * sigma_t**2))
            wavelet -= np.mean(wavelet)
            norm = np.sqrt(np.sum(np.abs(wavelet) ** 2))
            if norm > 0:
                wavelet /= norm
            cwt_matrix[row] = fftconvolve(values, wavelet.conj()[::-1], mode="same")
        method = f"FFT Morlet fallback, {WAVELET_FALLBACK_N_CYCLES:g} cycles"

    power = np.abs(cwt_matrix) ** 2
    order = np.argsort(freqs)
    freqs = np.asarray(freqs)[order]
    power = power[order]
    if WAVELET_POWER_SMOOTH_SIGMA and WAVELET_POWER_SMOOTH_SIGMA > 0:
        power = gaussian_filter(power, float(WAVELET_POWER_SMOOTH_SIGMA))
    return freqs, power, method

wavelet_channels, wavelet_t_video_s, wavelet_lfp_uv, wavelet_sample_rate_hz = load_frequency_lfp_window(
    [WAVELET_CHANNEL],
    start_min=WAVELET_START_MIN,
    duration_s=WAVELET_DURATION_S,
)
wavelet_trace = wavelet_lfp_uv[0]
wavelet_freqs, wavelet_power, wavelet_method = cwt_power_like_ephys_functions(wavelet_trace, wavelet_sample_rate_hz)
wavelet_power_db = 10.0 * np.log10(wavelet_power + np.finfo(float).tiny)
vmin, vmax = np.nanpercentile(wavelet_power_db, [WAVELET_VMIN_PERCENTILE, WAVELET_VMAX_PERCENTILE])

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
axes[0].plot(wavelet_t_video_s / 60.0, wavelet_trace, color="black", lw=0.8)
axes[0].set_title(
    f"LFP channel {int(wavelet_channels[0])}, {wavelet_t_video_s[0] / 60:.2f}-{wavelet_t_video_s[-1] / 60:.2f} min"
)
axes[0].set_ylabel("LFP (uV, preprocessed)")
axes[0].grid(alpha=0.25)

pcm = axes[1].pcolormesh(
    wavelet_t_video_s / 60.0,
    wavelet_freqs,
    wavelet_power_db,
    shading="auto",
    cmap="magma",
    vmin=vmin,
    vmax=vmax,
)
axes[1].set_yscale("log")
axes[1].set_xlabel("Video time (min)")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_title(f"CWT power ({wavelet_method})")
fig.colorbar(pcm, ax=axes[1], label="Power (dB)")
plt.show()



using cached LFP frequency data: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
global median removal skipped for a single loaded channel
loaded frequency LFP: 1 channel(s) x 75,000 samples, 250.00 Hz, video time 25.00-30.00 min
